# Generative Models Finetuning
## Instructions on How to use the notebook

This notebook is thought to be able to perform the finetuning of both FLAN-t5 e GPT2 models, and to perform the evaluation of both the base and finetuned model, using both the corss encoder and deepseek scores, with a variable k.

This notebook can be runned all-in-one, or section per section. Whathever portion of the notebook you want to run, be sure to run anyway the Enviroment Setpup section and all the "Settings" section located before the code you want to run.

Before start, be sure correctly set all the paths in the Paths section of the notebook

## Enviroment setup

In this section, we prepare the enviroment of the notebook, by installing and importing libreries and creating global variables that will be useful later

### Libraries

In [ ]:
# === Install required packages (run once in your notebook environment) ===
%pip install rouge_score bert_score evaluate

# === Standard Libraries ===
import math                      # For mathematical operations (e.g., exp, log)
import random as rd              # For random sampling/shuffling
import re                        # For regular expressions
from itertools import chain      # For flattening lists and chaining iterables

# === Data Handling ===
import pandas as pd              # For handling tabular data
import numpy as np               # For numerical operations
from sklearn.model_selection import train_test_split  # For splitting datasets

# === Hugging Face Datasets ===
from datasets import load_dataset, Dataset  # For loading and creating datasets

# === PyTorch ===
import torch                     # Main PyTorch library
from torch.utils.data import DataLoader  # For batching and iterating over data

# === Hugging Face Transformers ===
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,  # Tokenizers and models
    GPT2Tokenizer, GPT2LMHeadModel,                              # GPT-2 specific classes
    Seq2SeqTrainer, Seq2SeqTrainingArguments,                    # Seq2Seq training tools
    Trainer, TrainingArguments,                                  # General Trainer API
    TextDataset, DataCollatorWithPadding,                        # For dataset formatting
    T5ForConditionalGeneration,T5Tokenizer,
    pipeline                                                     # For easy model inference
)

# === Evaluation Libraries ===
import evaluate                                # Unified evaluation library from Hugging Face
from evaluate import load                      # Explicit import of metric loader
from sklearn.metrics import (                  # For traditional classification metrics
    accuracy_score, precision_score, recall_score, f1_score
)

# === PEFT (Parameter-Efficient Fine-Tuning) ===
from peft import LoraConfig, get_peft_model, TaskType, PeftModel  # For LoRA-based fine-tuning

# === Utility Libraries ===
from tqdm import tqdm                          # For progress bars

# === Sentence Transformers ===
from sentence_transformers import SentenceTransformer, util  # For embedding-based similarity


### Paths

In [ ]:
#Data path
dataset_path = "FreedomIntelligence/RAG-Instruct"

#Scores Paths
xencoder_train_path = '/xenc-scores-distilroberta/xenc_scores_test-stsb-distilroberta-base.npy'
xencoder_test_path = '/xenc-scores-distilroberta/xenc_scores_test-stsb-distilroberta-base.npy'

deepseek_train_path = ''
deepseek_test_path = ''

#Model download paths
flan_t5_path = "google/flan-t5-base"
gpt2_path = "gpt2"

#Model save path
flan_t5_save_path = ""
gpt2_save_path = ""

#Output path
output_path = "/ModelOutputs/"

#Metrics path
metrics_path = "/Metrics/"

### Download Models

In [ ]:
#Download FLAN t5
t5_model = AutoModelForSeq2SeqLM.from_pretrained(flan_t5_path)
t5_tokenizer = AutoTokenizer.from_pretrained(flan_t5_path)

#Download GPT2
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(gpt2_path)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # Set pad_token for batching
gpt2_model = GPT2LMHeadModel.from_pretrained(gpt2_path).to("cuda")

### Download Dataset

In [ ]:
ds_train = load_dataset(dataset_path, split="train[:80%]")
ds_test = load_dataset(dataset_path, split="train[80%:]")

### Load Rankings

In [ ]:
xencoder_rankings_train = np.load(xencoder_train_path)
xencoder_rankings_test = np.load(xencoder_test_path)

deepseek_rankings_train = np.load(deepseek_train_path)
deepseek_rankings_test = np.load(deepseek_test_path)

## Dataset Building

In this section we build the dataset for the fine tuning task. The dataset will contain only the top-k documents according to the scores obtained from the retrival section.

### Dataset Building Settings

In [ ]:
ranking_source = "" # Write one between "xencoder" and "deepseek". This will determine the source of the score to use.

k = 3 # Choose how many documents (top-k) will be included in the context
treshold = 1e-6 # Choose the minimum score a document must have to be included in the context

In [ ]:
rankings_train = None
rankings_test = None

if (ranking_source == "xencoder"):
    rankings_train = xencoder_rankings_train
    rankings_test = xencoder_rankings_test

elif (ranking_source == "deepseek"):
    rankings_train = deepseek_rankings_train
    rankings_test = deepseek_rankings_test

else:
    print("WARNING! Ranking source not aviable")

### Build Dataset Function

In [ ]:
def build_df(ds, doc_rankings, k = 3, treshold = 1e-6, debug_mode = False):
    docs = [d for d in ds['documents']]
    questions = [q for q in ds['question']]
    answers = [a for a in ds['answer']]
    data = []

    for i, (q, a) in enumerate(zip(questions, answers)):
        ranked_indices = [int(t[1]) for t in doc_rankings[i][:k] if float(t[0]) > treshold]
        top_docs = [docs[i][idx] for idx in ranked_indices]
        data.append({
            'question': q,
            'answer': a,
            'topk_documents': top_docs
        })
        if(debug_mode):
            print(i)
            print("Question: " + q + "\n")
            print("Documents:\n")
            for d in top_docs:
                print(d + "\n")
            print("Answer: " + a + "\n")
    df = pd.DataFrame(data)
    topk_df = Dataset.from_pandas(df)
    return topk_df

### Test build_df

In [ ]:
debug_range = range(0,10)
debug_dataset = ds_train[0:10]
debug_scores = rankings_train[0:10]
topk_ds_debug = build_df(debug_dataset, debug_scores)
debug_dataset_questions = ds_train['question'][0:10]
debug_dataset_answers = ds_train['answer'][0:10]
debug_dataset_documents = ds_train['documents'][0:10]
i = 0
print(str(i) + "\n")
print("Dataset Question:\n" + debug_dataset_questions[i] + "\n")
print("DF Question:\n" + topk_ds_debug[i]['question'] + "\n")
print("All Documents:\n")
for i,d in enumerate(debug_dataset_documents[i]):
    print(str(i) + ": " + d + "\n")
print("Scores:\n")
for s in debug_scores[0]:
    print(s)
print("\nTop K Documents:\n")
for d in topk_ds_debug[0]['topk_documents']:
    print(d + "\n")
print("Dataset Answer: " + debug_dataset_answers[0] + "\n")
print("DS Answer: " + topk_ds_debug[0]['answer'] + "\n")

### Build And Split Datasets

In [ ]:
topk_ds_train = build_df(ds_train, rankings_train)
topk_ds_testval = build_df(ds_test, rankings_test)

topk_ds_split = topk_ds_testval.train_test_split(
    test_size=0.25,
    shuffle=True,
    seed=42
)

topk_ds_val = topk_ds_split["train"]
topk_ds_test = topk_ds_split["test"] 

## Finetuning
In this section, we will finetune one of the models using the dataset created before

### Finetuning Setting

In [ ]:
model_to_use = "" # Write one between "FLAN t5" and "GPT2". This will determine the model used for training.

if (model_to_use == "FLAN t5"):
     
    model = t5_model
    tokenizer = t5_tokenizer

elif (model_to_use == "GPT2"):

    model = gpt2_model
    tokenizer = gpt2_tokenizer


### Prepocessing Functions

In [ ]:
def preprocess_function_T5(examples, tokenizer, max_source_length=512, max_target_length=512):
    inputs = []
    for question, documents in zip(examples["question"], examples["topk_documents"]):
        context = " ".join(documents)
        #inputs.append(f"question: {question} context: {context}")

        inputs.append(f"Question:\n{question}\n\nContext:\n{context}\n\nAnswer: ")
    
    model_inputs = tokenizer(
        inputs, 
        max_length=max_source_length, 
        truncation=True, 
        padding="max_length"
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["answer"], 
            max_length=max_target_length, 
            truncation=True, 
            padding="max_length"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:

def preprocess_function_GPT2(examples, tokenizer):
    inputs = []
    for question, docs, answer in zip(examples["question"], examples["topk_documents"], examples["answer"]):
        context = " ".join(chain.from_iterable(docs)) if isinstance(docs[0], list) else " ".join(docs)
        if isinstance(answer, list):
            answer = " ".join(map(str, answer))
        else:
            answer = str(answer)

        prompt = f"### Question:\n{question}\n\n### Context:\n{context}\n\n### Answer:\n"
        input_text = prompt + answer
        inputs.append(input_text)

    tokenized = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

In [ ]:
def preprocess_function(model, examples, tokenizer, max_source_length=512, max_target_length=512):
    if(model == "FLAN t5"):
        return preprocess_function_T5(examples, tokenizer, max_source_length, max_source_length)
    if(model == "GPT2"):
        return preprocess_function_GPT2(examples, tokenizer)
    
    print("ERROR! Model not found! Returning None")

    return None

In [ ]:
lora_config = None
if (model_to_use == "FLAN t5"):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=8, 
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q", "k", "v"]#, "o", "Linear"]
    )

elif (model_to_use == "GPT2"):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["c_attn"],  # <- For GPT-2 attention layer
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

In [ ]:
def create_lora_model(model):
    return get_peft_model(model, lora_config)

In [ ]:
def pd_to_hf_ds(dataset, tokenizer):
    if isinstance(dataset, pd.DataFrame):
        dataset = Dataset.from_pandas(dataset)
    
    tokenized_dataset = dataset.map(lambda x: preprocess_function(x, tokenizer), batched=True)
    return tokenized_dataset

### Create Trainer

In [ ]:
if (model_to_use == "FLAN t5"): 
    model = create_lora_model(model)
    model.print_trainable_parameters()

    topk_hf_train = pd_to_hf_ds(topk_ds_train, tokenizer)
    topk_hf_val = pd_to_hf_ds(topk_ds_val, tokenizer)

    training_args = Seq2SeqTrainingArguments(
        output_dir=flan_t5_save_path,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=2,
        predict_with_generate=True,
        bf16=True, #fp16
        gradient_accumulation_steps=4, #4
        report_to="tensorboard",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=topk_hf_train,
        eval_dataset=topk_hf_val,
        tokenizer=tokenizer,
    )

elif (model_to_use == "GPT2"):
    model = create_lora_model(model)
    model.print_trainable_parameters()

    topk_hf_train = pd_to_hf_ds(topk_ds_train, tokenizer)
    topk_hf_val = pd_to_hf_ds(topk_ds_val, tokenizer)

    training_args = TrainingArguments(
        output_dir=gpt2_save_path,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        learning_rate=3e-4,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=3,
        fp16=True,
        gradient_accumulation_steps=4,
        report_to="tensorboard",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=topk_hf_train,
        eval_dataset=topk_hf_val,
        tokenizer=tokenizer,
    )

### Train And Save Adapter

In [ ]:
trainer.train()

model_name = model_to_use + "-lora-qa-" + ranking_source + "k=" + str(k)
model.save_pretrained(model_name)
tokenizer.save_pretrained(model_name)

## Model Evaluation

This section perform model evaluation and compute metrics to measure the models performance.

### Load Model Adapter

In [ ]:
# model_name = ""  # use this instruction if you are running this part of the notebook without running the previous one to specify the model you want to load

if (model_to_use == "FLAN t5"): 
    base_model = t5_model
    adapter_path = gpt2_save_path + model_name

if (model_to_use == "GPT2"):
    base_model = gpt2_model
    adapter_path = gpt2_save_path + model_name

ft_model = PeftModel.from_pretrained(base_model, adapter_path).to("cuda")

### Evaluate Model Functions

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    return text

def generate_answer(model, tokenizer, question, context, max_length=512):
    # Prompt in stile naturale adatto a GPT-2
    prompt = f"{context}\n\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Estrai solo la parte dopo "Answer:"
    if "Answer:" in decoded:
        return decoded.split("Answer:")[-1].strip()
    else:
        return decoded.strip()



def evaluate_model_casualLM(model, tokenizer, dataset, max_samples=5, print_examples=True):

    model.eval()
    dataset = dataset.select(range(min(len(dataset), max_samples)))
    model_output = []
    losses = []

    for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
        single_output = []
        question = example["question"]
        context_list = example["topk_documents"]
        context = "\n".join(context_list)
        reference = example["answer"]

        # Calcolo della loss (perplexity)
        full_target_text = f"{context}\n\nQuestion: {question}\nAnswer: {reference}"
        encoding = tokenizer(full_target_text, return_tensors="pt", truncation=True, max_length=512)
        input_ids = encoding["input_ids"].to(model.device)
        attention_mask = encoding["attention_mask"].to(model.device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            losses.append(outputs.loss.item())

        # Generazione della predizione
        prediction = generate_answer(model, tokenizer, question, context)

        single_output.append(question)
        single_output.append(context)
        single_output.append(reference)
        single_output.append(prediction)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference}")

        model_output.append(single_output)

    average_loss = sum(losses) / len(losses) if losses else float("inf")
    return model_output, average_loss


In [ ]:
def evaluate_model_seq2seq(model, tokenizer, dataset, max_samples=5, print_examples=True):

    model.eval()
    dataset = dataset.select(range(min(len(dataset), max_samples)))
    
    model_output = []
    losses = []

    for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
        single_output = []
        question = example["question"]
        context_list = example["topk_documents"]
        context = "\n".join(context_list)
        reference = example["answer"]

        # Input e target per seq2seq
        input_text = f"{context}\n\nQuestion: {question}"
        target_text = reference

        # Tokenizzazione
        input_encoding = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(model.device)
        target_encoding = tokenizer(target_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(model.device)

        with torch.no_grad():
            outputs = model(
                input_ids=input_encoding["input_ids"],
                attention_mask=input_encoding["attention_mask"],
                labels=target_encoding["input_ids"]
            )
            losses.append(outputs.loss.item())

        # Generazione predizione
        output_ids = model.generate(
            input_encoding["input_ids"],
            attention_mask=input_encoding["attention_mask"],
            max_new_tokens=128,
            num_beams=4,
            do_sample=False
        )
        prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Salvataggio risultati
        single_output.extend([question, context, reference, prediction])
        model_output.append(single_output)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference}")

    average_loss = sum(losses) / len(losses) if losses else float("inf")
    return model_output, average_loss


In [ ]:
def evaluate_model(model_name, model, tokenizer, dataset, max_samples=5, print_examples=False, architecture = "seq2seq"):
    
    if(architecture == "seq2seq"):
        model_output, average_loss = evaluate_model_seq2seq(model, tokenizer, dataset, max_samples, print_examples)

    if(architecture == "casualLM"):
        model_output, average_loss = evaluate_model_casualLM(model, tokenizer, dataset, max_samples, print_examples)

    output_save_path =output_path + "model_output_" + model_name + ".npy"
    np.save(output_save_path, {
    "output": model_output,
    "loss": average_loss
    })
    return output_save_path
 

### Evaluate Output Function

In [ ]:
def evaluate_output(data, tokenizer, print_examples=False):

    model_output = data["output"]
    eval_loss = data["loss"]
    
    bertscore = evaluate.load("bertscore")
    rouge = evaluate.load("rouge")
    em = evaluate.load("exact_match")
    bleu = evaluate.load("bleu")
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    predictions = []
    references = []
    faithfulness_scores = []
    relevance_scores = []
    length_ratios = []
    diversity_set = set()

    for i, example in enumerate(model_output):
        question = example[0]
        context = example[1]  # single string with all docs concatenated
        reference = example[2]
        prediction = example[3]

        # Faithfulness: sim(context, prediction)
        doc_emb = embed_model.encode(context, convert_to_tensor=True)
        pred_emb = embed_model.encode(prediction, convert_to_tensor=True)
        faithfulness_scores.append(util.cos_sim(pred_emb, doc_emb).item())

        # Relevance: sim(question, prediction)
        question_emb = embed_model.encode(question, convert_to_tensor=True)
        relevance_scores.append(util.cos_sim(pred_emb, question_emb).item())

        # BLEU (calcolato in seguito)
        predictions.append(prediction)
        references.append(reference)

        # Token Length Ratio
        ref_tokens = tokenizer.encode(reference, add_special_tokens=False)
        pred_tokens = tokenizer.encode(prediction, add_special_tokens=False)
        length_ratio = len(pred_tokens) / len(ref_tokens) if len(ref_tokens) > 0 else 0
        length_ratios.append(length_ratio)

        # Diversity (vocab size / total tokens)
        diversity_set.update(pred_tokens)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference}")

    # Metriche
    bertscore_f1 = bertscore.compute(predictions=predictions, references=references, lang="en")["f1"]
    rouge_score = rouge.compute(predictions=predictions, references=references)
    em_score = em.compute(predictions=predictions, references=references)["exact_match"]
    bleu_score = bleu.compute(predictions=predictions, references=references)["bleu"]
    perplexity_score = math.exp(eval_loss)

    token_count = sum(len(tokenizer.encode(p, add_special_tokens=False)) for p in predictions)
    diversity = len(diversity_set) / token_count if token_count > 0 else 0

    return {
        "RougeL": rouge_score["rougeL"],
        "EM": em_score,
        "BERTScore_F1": sum(bertscore_f1) / len(bertscore_f1),
        "BLEU": bleu_score,
        "Perplexity": perplexity_score,
        "Faithfulness": sum(faithfulness_scores) / len(faithfulness_scores),
        "Relevance": sum(relevance_scores) / len(relevance_scores),
        "Token_Length_Ratio": sum(length_ratios) / len(length_ratios),
        "Diversity": diversity
    }


### Evaluation Settings

In [ ]:
model = None #select one between base_model or ft_model

print(model.name_or_path)

### Evaluate Model

In [ ]:
if (model_to_use == "FLAN t5"): 
    architecture = "seq2seq"

if (model_to_use == "GPT2"):
    architecture = "casualLM"

path = evaluate_model(model, tokenizer, topk_ds_test,architecture=architecture)

### Compute And Save Metrics

In [ ]:
data = np.load(path)
metrics = evaluate_output(data, tokenizer)

print("\nMetrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

metrics_save_path = metrics_path + "metrics_" + model.name_or_path + ".npy"
np.save(metrics_save_path, {
"RougeL": metrics["RougeL"],
"EM": metrics["EM"],
"BERTScore_F1": metrics["BERTScore_F1"],
"BLEU": metrics["BLEU"],
"Perplexity": metrics["Perplexity"],
"Faithfulness": metrics["Faithfulness"],
"Relevance": metrics["Relevance"],
"Token_Length_Ratio": metrics["Token_Length_Ratio"],
"Diversity": metrics["Diversity"]
})